# Basic Question & Answer Generation from a FileSet

This notebook demonstrates the simplest way to generate question-answer pairs from a FileSet. Documents are chunked into seeds, then questions and labels are generated in a single step using `QuestionAndLabelGenerator`.

**Prerequisite**: Run `01_create_fileset.ipynb` first to create a FileSet and upload documents.

In [8]:
%pip install ../.. python-dotenv -q

from IPython.display import clear_output
clear_output()

## Set up the client

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/sign-up?redirect=/api) to get your API key and **$50 of free credits**.

- **Google Colab**: Go to the Secrets section (key icon in left sidebar) and add a secret named `LIGHTNINGROD_API_KEY`
- **Local Jupyter**: Set the `LIGHTNINGROD_API_KEY` environment variable, or you'll be prompted to enter it

In [9]:
from dotenv import load_dotenv
from lightningrod import LightningRod
from lightningrod.utils import config

load_dotenv()
api_key = config.get_config_value("LIGHTNINGROD_API_KEY")

lr = LightningRod(api_key=api_key)

## Configure FileSet ID

Paste the FileSet ID from notebook 1 below.

In [10]:
fileset_id = "af5e701c-94dd-460f-9864-70891f880443"

## Configure the Question Pipeline

- **`FileSetSeedGenerator`** chunks the documents in your FileSet into seeds (text passages)
- **`QuestionAndLabelGenerator`** generates questions and answers in a single step — the simplest way to produce labeled Q&A pairs

In [11]:
from lightningrod import (
    QuestionPipeline,
    FileSetSeedGenerator,
    QuestionAndLabelGenerator,
    FreeResponseAnswerType,
)

answer_type = FreeResponseAnswerType()

pipeline = QuestionPipeline(
    seed_generator=FileSetSeedGenerator(
        file_set_id=fileset_id,
        chunk_size=2000,
        chunk_overlap=200,
    ),
    question_generator=QuestionAndLabelGenerator(
        questions_per_seed=2,
        answer_type=answer_type,
        instructions=(
            "Generate questions about the financial metrics, business events, "
            "and forward guidance in these quarterly investor reports. Questions should be "
            "specific and verifiable from the report content."
        ),
    ),
)

## Run the Pipeline

In [12]:
dataset = lr.transforms.run(
    pipeline,
    name="FileSet - Basic QA",
)
print(f"Dataset: {dataset.id}")
print(f"Rows: {dataset.num_rows}")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Pipeline Completed                                                                                          │
│                                                                                                                 │
│    Job ID:           ea8451e5-a9e9-464c-b8b4-958df2bf77f9                                                       │
│                                                                                                                 │
│    Total cost: $0.01                                                                                            │
│                                                                                                                 │
│  ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━┳━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━┓  │
│  ┃ Step                                  ┃ Progress               ┃  In ┃ Out ┃ Rejected ┃ Errors ┃ Duration ┃  │
│  ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━╇━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━┩  │
│  │ FileSetSeedGeneratorTransform         │ Complete               │  10 │  10 │        0 │      0 │       8s │  │
│  │ QuestionAndLabelGeneratorTransform    │ Complete               │  10 │  20 │        0 │      0 │       3s │  │
│  └───────────────────────────────────────┴────────────────────────┴─────┴─────┴──────────┴────────┴──────────┘  │
│                                                                                                                 │
│    View full details:                                                                                           │
│  ]8;id=508443;https://dashboard.lightningrod.ai/?redirect=/datasets/1a7d0c8a-9830-453c-a9df-242e1fd294f4\https://dashboard.lightningrod.ai/?redirect=/datasets/1a7d0c8a-9830-453c-a9df-242e1fd294f4]8;;\                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Dataset: 1a7d0c8a-9830-453c-a9df-242e1fd294f4
Rows: 20


> **Note:** This can take a few minutes to complete processing.

## View the Results

In [13]:
%pip install pandas -q

from IPython.display import clear_output
clear_output()

In [14]:
import pandas as pd

samples = dataset.download()
rows = dataset.flattened()
df = pd.DataFrame(rows)

print(f"Generated {dataset.num_rows} samples ({dataset.valid_count() / dataset.num_rows * 100:.1f}% valid)\n")

cols = ["question_text", "label", "label_confidence", "is_valid"]
df[[c for c in cols if c in df.columns]]

Generated 20 samples (100.0% valid)



,question_text,label,label_confidence,is_valid
0,In which quarter does APEX Technologies Inc. a...,Q3 2025,1.0,True
1,"According to the Q4 2024 reporting, what is th...","FY2025 revenue is guided at $7.0B-$7.4B, with ...",1.0,True
2,What is the expected completion timeline and t...,The $400M data center expansion is on schedule...,1.0,True
3,What was the year-over-year revenue growth per...,The RaaS segment grew 107% YoY and is projecte...,1.0,True
4,What was the total revenue generated by Vangua...,"$450 million, which represented 27% of total r...",1.0,True
5,"According to the Q1 2025 report, what is the e...","Q2 2025 revenue is expected at $1.82B-$1.90B, ...",1.0,True
6,What is the updated full-year 2024 revenue gui...,$6.2B-$6.4B,1.0,True
7,When does APEX Technologies Inc. expect to clo...,Q1 2025,1.0,True
8,What was the total revenue contribution from c...,$180M,1.0,True
9,When does Vanguard Industries Inc. expect to c...,Early Q4 2024,1.0,True
